# Setup: LiteLLM + OpenRouter

LiteLLM routes any model whose id starts with `openrouter/` to OpenRouter, using `OPENROUTER_API_KEY` from `.env`.
Id format: `openrouter/<vendor>/<model>`, e.g. `openrouter/openai/gpt-4o-mini` or `openrouter/anthropic/claude-sonnet-4.5`.
Browse ids at https://openrouter.ai/models.

In [ ]:
from scratch_agents.env import key_status
from scratch_agents.models import Model, Tier, get_client, list_models

MODEL = Model.GPT_6_LUNA
print(key_status())
list_models(Tier.BUDGET)

## 1. Plain LiteLLM call

In [ ]:
import litellm

resp = litellm.completion(model=MODEL.value, messages=[{"role": "user", "content": "Say hi in 5 words."}])
print(resp.choices[0].message.content)
print(resp.usage)

## 2. The repo's `LlmClient` (async, what every chapter uses)

In [ ]:
client = get_client(MODEL)
print(await client.ask("What is 2 + 2? Answer with one word."))

## 3. Tool calling through OpenRouter

In [ ]:
from scratch_agents import Agent
from scratch_agents.tools import calculator

agent = Agent(model=get_client(MODEL), tools=[calculator], instructions="Use tools for math.")
result = await agent.run("What is 1234 * 5678?")
print(result.output)

## Using OpenRouter in the other chapters

Start any notebook with `from scratch_agents.env import require` (loads `.env`, checks the key) and use
`get_client(Model.X)` instead of `LlmClient(model="gpt-...")`.
Not everything goes through LiteLLM: CH06 uses OpenAI directly for ChromaDB embeddings, so it still needs `OPENAI_API_KEY`.